In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from roboflow import Roboflow
from ultralytics import YOLO

In [2]:
load_dotenv()
ROBOFLOW_KEY = os.getenv("ROBOFLOW_KEY")

In [3]:
rf = Roboflow(api_key=ROBOFLOW_KEY)
project = rf.workspace("goat-yjce4").project("road-report-yolo-u6eo6")
version = project.version(1)
dataset = version.download(
    "yolov11",
    location="../datasets/pothole-data"
)

loading Roboflow workspace...
loading Roboflow project...


In [4]:
model = YOLO("yolo11n.pt")

In [5]:
PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models" / "yolov11n"

In [6]:
model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=512,
    batch=8,
    project=str(MODEL_DIR.parent),
    name=MODEL_DIR.name,
    patience=10,
    save=True,
    val=True,
    plots=True,
)

Ultralytics 8.4.196  Python-3.13.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\vallu\Desktop\Road Report\datasets\pothole-data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000022703F0C670>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [8]:
model = YOLO(str(MODEL_DIR / "weights" / "best.pt"))

metrics = model.val(
    data=f"{dataset.location}/data.yaml",
    split="test",
    project=str(MODEL_DIR.parent),
    name = "test",
    imgsz=512,
    batch=8)

print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

Ultralytics 8.4.196  Python-3.13.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.1±0.0 ms, read: 161.9±42.3 MB/s, size: 49.1 KB)
val: Scanning C:\Users\vallu\Desktop\Road Report\datasets\pothole-data\test\labels.cache... 67 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 67/67 21.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 1.3s/it 12.0s0.2s3
                   all         67        158      0.836      0.684      0.793      0.473
Speed: 3.5ms preprocess, 9.9ms inference, 0.0ms loss, 3.4ms postprocess per image
Results saved to C:\Users\vallu\Desktop\Road Report\models\test
Precision: 0.8357
Recall: 0.6835
mAP50: 0.7932
mAP50-95: 0.4729
